In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── Comparing activation functions ────────────────────────────────────
x = torch.linspace(-3, 3, 100)

sigmoid_out  = torch.sigmoid(x)        # output always in (0, 1)
tanh_out     = torch.tanh(x)           # output in (-1, 1), zero-centred
relu_out     = F.relu(x)              # max(0, x) — no saturation for x>0
leaky_out    = F.leaky_relu(x, 0.01)  # small slope for x<0

# ── Gradients (derivative at x = 2.0 and x = -2.0) ───────────────────
for name, fn in [('sigmoid', torch.sigmoid), ('tanh', torch.tanh),
                 ('relu', F.relu), ('leaky_relu', lambda x: F.leaky_relu(x,0.01))]:
    for val in [2.0, -2.0]:
        x_t = torch.tensor([val], requires_grad=True)
        fn(x_t).backward()
        print(f'{name:12s}  x={val:5.1f}  gradient={x_t.grad.item():.4f}')

# sigmoid   x= 2.0  gradient=0.1049   ← already small
# sigmoid   x=-2.0  gradient=0.1049   ← same (symmetric)
# tanh      x= 2.0  gradient=0.0707   ← smaller than sigmoid here
# tanh      x=-2.0  gradient=0.0707
# relu      x= 2.0  gradient=1.0000   ← constant, never saturates
# relu      x=-2.0  gradient=0.0000   ← dead zone
# leaky_relu x= 2.0 gradient=1.0000
# leaky_relu x=-2.0 gradient=0.0100   ← small but non-zero

# ── Dead ReLU demonstration ───────────────────────────────────────────
torch.manual_seed(0)
layer = nn.Linear(10, 5)

# Force all pre-activations to be very negative by using a bad init
with torch.no_grad():
    layer.weight.fill_(-1.0)
    layer.bias.fill_(-5.0)

x_in = torch.randn(4, 10)   # positive inputs
out  = F.relu(layer(x_in))
print(f'Dead neurons: {(out == 0).all(dim=0).sum().item()} / {out.shape[1]}')
# All 5 neurons are dead — no gradient will flow back through them


In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ── Use fake data for a fast demo ─────────────────────────────────────
# This avoids a slow CIFAR-10 download while keeping the same pipeline
# for computing dataset statistics and applying normalization.
fake_train = datasets.FakeData(
    size=2000,
    image_size=(3, 32, 32),
    num_classes=10,
    transform=transforms.ToTensor(),
)
loader = DataLoader(fake_train, batch_size=256, shuffle=False)

# Accumulate mean and std across the synthetic training set
mean = torch.zeros(3)
std  = torch.zeros(3)
n_batches = 0
for imgs, _ in loader:
    mean += imgs.mean(dim=[0, 2, 3])
    std  += imgs.std(dim=[0, 2, 3])
    n_batches += 1
mean /= n_batches
std  /= n_batches
print(f'FakeData mean: {mean.tolist()}')
print(f'FakeData std:  {std.tolist()}')

# ── Apply normalization using the synthetic dataset statistics ────────
normalise = transforms.Normalize(mean=mean.tolist(), std=std.tolist())

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    normalise,
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    normalise,
])

# ── Quick check: after normalization, data should be ~zero-centred ────
sample_batch = next(iter(DataLoader(
    datasets.FakeData(
        size=256,
        image_size=(3, 32, 32),
        num_classes=10,
        transform=train_transform,
    ),
    batch_size=128,
)))[0]
print(f'After normalisation — mean: {sample_batch.mean():.4f}, std: {sample_batch.std():.4f}')
# Should be close to 0.0 and 1.0 for the synthetic pipeline.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── Visualising activation statistics under different initialisations ──
torch.manual_seed(0)
N_LAYERS = 10
DIM      = 512

def forward_stats(init_fn, activation):
    """Run a forward pass and report mean/std of activations per layer."""
    x = torch.randn(256, DIM)
    for _ in range(N_LAYERS):
        W = torch.empty(DIM, DIM)
        init_fn(W)
        x = activation(x @ W)
    return x.mean().item(), x.std().item()

configs = [
    ('σ=0.01 + tanh',  lambda W: nn.init.normal_(W, std=0.01),  torch.tanh),
    ('σ=0.05 + tanh',  lambda W: nn.init.normal_(W, std=0.05),  torch.tanh),
    ('Xavier + tanh',  nn.init.xavier_normal_,                  torch.tanh),
    ('Kaiming + ReLU', nn.init.kaiming_normal_,                 F.relu),
    ('σ=0.01 + ReLU',  lambda W: nn.init.normal_(W, std=0.01),  F.relu),
]

print(f'{"Config":25s}  mean      std')
print('-' * 50)
for name, init, act in configs:
    mean, std = forward_stats(init, act)
    print(f'{name:25s}  {mean:+.4f}   {std:.4f}')

# Expected output (approximately):
# σ=0.01 + tanh          mean ≈ 0       std ≈ 0.0001  ← vanished
# σ=0.05 + tanh          mean ≈ 0       std ≈ 1.0     ← saturated, flat tanh
# Xavier + tanh          mean ≈ 0       std ≈ 0.8     ← healthy
# Kaiming + ReLU         mean ≈ 0.4     std ≈ 0.7     ← healthy
# σ=0.01 + ReLU          mean ≈ 0       std ≈ 0.0     ← vanished

# ── Using PyTorch's built-in initialisation ───────────────────────────
model = nn.Sequential(
    nn.Linear(512, 256), nn.ReLU(),
    nn.Linear(256, 128), nn.ReLU(),
    nn.Linear(128, 10),
)

# Apply Kaiming initialisation to all Linear layers
for m in model.modules():
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
        nn.init.zeros_(m.bias)

# Apply Xavier for tanh layers
for m in model.modules():
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        nn.init.zeros_(m.bias)


In [ ]:
import torch
import torch.nn as nn

# ── Demonstrating exploding vs vanishing with raw matrix multiplications
# (no activation, to isolate the initialisation effect)
torch.manual_seed(42)
L = 10       # number of layers
D = 64       # dimension
x = torch.randn(1, D)

for scale, name in [(1.5, 'scale=1.5 (exploding)'),
                    (0.01, 'scale=0.01 (vanishing)'),
                    (1.0/D**0.5, 'scale=1/sqrt(D) (stable)')]:
    out = x.clone()
    for _ in range(L):
        W   = scale * torch.randn(D, D)
        out = out @ W
    print(f'{name}: output norm = {out.norm():.2e}')

# scale=1.5 (exploding):         output norm ≈ 1e+10  ← often nan in practice
# scale=0.01 (vanishing):        output norm ≈ 1e-10  ← gradient dead
# scale=1/sqrt(D) (stable):      output norm ≈ 1e+00  ← healthy

In [ ]:
import torch
import torch.nn as nn

# ── Batch normalisation: mechanics and PyTorch API ────────────────────

# ── For fully connected layers: nn.BatchNorm1d ────────────────────────
# Input shape: (batch, features)
bn1d = nn.BatchNorm1d(num_features=64)   # one γ and β per feature

x_fc = torch.randn(32, 64)   # batch of 32, 64 features each
y_fc = bn1d(x_fc)
print(f'BN1d output: mean={y_fc.mean():.4f}, std={y_fc.std():.4f}')
# Should be approximately 0.0 and 1.0 (before γ,β are updated)

# ── For convolutional layers: nn.BatchNorm2d ──────────────────────────
# Input shape: (batch, channels, H, W)
# Normalises across batch AND spatial dimensions, per channel
bn2d = nn.BatchNorm2d(num_features=32)   # one γ and β per channel

x_conv = torch.randn(8, 32, 14, 14)   # 8 images, 32 channels, 14×14
y_conv = bn2d(x_conv)
print(f'BN2d output: mean={y_conv.mean():.4f}, std={y_conv.std():.4f}')

# ── Standard CNN block: Conv → BN → ReLU ─────────────────────────────
class ConvBNReLU(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, padding=1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size, padding=padding, bias=False)
        # bias=False because BN has its own bias (β)
        self.bn   = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))

# ── CRITICAL: training vs eval mode ──────────────────────────────────
model = nn.Sequential(
    ConvBNReLU(3, 32), ConvBNReLU(32, 64),
    nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, 10)
)

x = torch.randn(4, 3, 32, 32)

# Training mode: uses batch statistics, updates running averages
model.train()
out_train = model(x)
print(f'Train mode output shape: {out_train.shape}')

# Eval mode: uses stored running averages (fixed, deterministic)
model.eval()
with torch.no_grad():
    out_eval = model(x)
print(f'Eval mode output shape:  {out_eval.shape}')

# These will differ if running averages haven't warmed up yet:
print(f'Max difference: {(out_train - out_eval).abs().max():.4f}')

# ── Inspecting the learnable parameters ───────────────────────────────
bn = nn.BatchNorm2d(8)
print(f'gamma (weight) shape: {bn.weight.shape}')  # (8,) — one per channel
print(f'beta  (bias)   shape: {bn.bias.shape}')    # (8,) — one per channel
print(f'Running mean   shape: {bn.running_mean.shape}')  # (8,) — not a param


In [ ]:
import torch
import torch.nn as nn
from matplotlib import pyplot as plt

# ── Demonstrating that BN stabilises training ─────────────────────────
# We compare a deep network with and without batch normalisation

def build_deep_net(use_bn, depth=10, width=8):
    layers = []
    in_ch  = 3
    for _ in range(depth):
        layers.append(nn.Conv2d(in_ch, width, 3, padding=1, bias=not use_bn))
        if use_bn:
            layers.append(nn.BatchNorm2d(width))
        layers.append(nn.ReLU(inplace=True))
        in_ch = width
    layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(width, 10)]
    return nn.Sequential(*layers)

torch.manual_seed(0)
X = torch.randn(8, 3, 32, 32)
y = torch.randint(0, 10, (8,))

plt.figure(figsize=(8, 4))

for use_bn in [False, True]:
    model = build_deep_net(use_bn=use_bn)
    # Use a high learning rate to stress-test
    opt   = torch.optim.SGD(model.parameters(), lr=0.1)
    loss_fn = nn.CrossEntropyLoss()

    losses = []
    for step in range(200):
        opt.zero_grad()
        loss = loss_fn(model(X), y)
        if torch.isnan(loss): losses.append(float('nan')); break
        loss.backward()
        # Gradient norm tells us if gradients are healthy
        grad_norm = sum(p.grad.norm() for p in model.parameters() if p.grad is not None)
        opt.step()
        losses.append(loss.item())

    label = 'WITH BN' if use_bn else 'WITHOUT BN'
    converged = not any(torch.isnan(torch.tensor(losses)))
    print(f'{label}: final loss={losses[-1]:.3f}, converged={converged}')
    plt.plot(losses, label=label)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Training Loss Comparison')
plt.legend()
plt.show()
# WITHOUT BN: likely NaN (diverged with lr=0.1)
# WITH BN:    converges stably


In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ── A well-configured CNN incorporating all four techniques ───────────

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# 1. DATA PREPROCESSING: normalise with training-set statistics
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),  # zero-centre
])

class WellConfiguredCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            # 2. BATCH NORM: Conv → BN → ReLU pattern throughout
            # 4. ACTIVATION FUNCTION: ReLU everywhere
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),   # BN before activation
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16, 256, bias=False),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )

        # 3. WEIGHT INITIALISATION: Kaiming for ReLU layers
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.kaiming_normal_(m.weight, mode='fan_in',
                                        nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.ones_(m.weight)   # γ = 1 at init
                nn.init.zeros_(m.bias)    # β = 0 at init

    def forward(self, x):
        return self.classifier(self.features(x))

model = WellConfiguredCNN(num_classes=10)

# Count parameters
n = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n:,}')

# Verify activation statistics before training (should be ~N(0,1))
model.eval()
x = torch.randn(64, 3, 32, 32)   # simulate normalised batch
with torch.no_grad():
    # Hook to capture intermediate activations
    activations = {}
    def hook(name):
        def fn(module, input, output):
            activations[name] = output.detach()
        return fn
    model.features[2].register_forward_hook(hook('relu1'))  # after first ReLU
    model.features[5].register_forward_hook(hook('relu2'))  # after second ReLU
    _ = model(x)

for name, act in activations.items():
    print(f'{name}: mean={act.mean():.3f}, std={act.std():.3f}')
# Both should show healthy statistics (~zero mean, ~unit std at init)
